# 00 — Data audit (metadata only)

This notebook inspects the competition CSVs only. It does **not** read DICOM pixel data or download the dataset. Run it after adding the RSNA competition data through Kaggle's **Add Input** panel.

The audit is descriptive: it does not assume missing labels are negative, and it does not train a model.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

TARGETS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture',
]

# Find the CSVs wherever Kaggle mounted the competition input.
roots = [Path('/kaggle/input'), Path.cwd()]
wanted = {'train.csv', 'train_series.csv', 'test.csv', 'test_series.csv', 'sample_submission.csv'}
found = {}
for root in roots:
    if root.exists():
        for p in root.rglob('*.csv'):
            if p.name in wanted and p.name not in found:
                found[p.name] = p

print('CSV files found:')
for name, path in sorted(found.items()):
    print(f'  {name}: {path}')
if 'train.csv' not in found or 'train_series.csv' not in found:
    raise FileNotFoundError('Could not find train.csv and train_series.csv. Use Kaggle Add Input to attach the competition dataset.')

train = pd.read_csv(found['train.csv'])
series = pd.read_csv(found['train_series.csv'])
print('\n=== SHAPES ===')
print('train:', train.shape, '| series:', series.shape)
print('\n=== TRAIN COLUMNS / DTYPES ===')
print(train.dtypes.to_string())
print('\n=== SERIES COLUMNS / DTYPES ===')
print(series.dtypes.to_string())

required_train = {'StudyInstanceUID', 'Report'}
missing_required = required_train - set(train.columns)
if missing_required:
    print('\nWARNING: expected train columns missing:', sorted(missing_required))
available_targets = [c for c in TARGETS if c in train.columns]
missing_targets = [c for c in TARGETS if c not in train.columns]
print('\nTarget columns present:', len(available_targets), '/', len(TARGETS))
if missing_targets:
    print('Target columns absent:', missing_targets)

print('\n=== IDENTIFIER INTEGRITY ===')
for name, df in [('train', train), ('train_series', series)]:
    if 'StudyInstanceUID' in df.columns:
        print(f'{name}: rows={len(df):,}, unique studies={df.StudyInstanceUID.nunique():,}, missing study IDs={df.StudyInstanceUID.isna().sum():,}')
if 'StudyInstanceUID' in train.columns:
    print('Duplicate study rows in train:', int(train.StudyInstanceUID.duplicated().sum()))
if {'StudyInstanceUID', 'SeriesInstanceUID'}.issubset(series.columns):
    print('Duplicate study-series pairs:', int(series.duplicated(['StudyInstanceUID', 'SeriesInstanceUID']).sum()))
    print('Series per study — describe:')
    print(series.groupby('StudyInstanceUID').size().describe().to_string())
    train_ids = set(train.StudyInstanceUID.dropna())
    series_ids = set(series.StudyInstanceUID.dropna())
    print('Train studies with no series metadata:', len(train_ids - series_ids))
    print('Series metadata studies absent from train.csv:', len(series_ids - train_ids))

print('\n=== LABEL COVERAGE AND PREVALENCE ===')
if available_targets:
    y = train[available_targets].apply(pd.to_numeric, errors='coerce')
    summary = pd.DataFrame({
        'non_missing': y.notna().sum(),
        'missing': y.isna().sum(),
        'unique_non_missing_values': y.nunique(dropna=True),
        'positive_count': (y == 1).sum(),
        'negative_count': (y == 0).sum(),
        'positive_rate_among_labelled': (y == 1).sum() / y.notna().sum().replace(0, np.nan),
    })
    print(summary.to_string())
    labelled_any = y.notna().any(axis=1)
    labelled_all = y.notna().all(axis=1)
    print(f'\nStudies with >=1 non-missing target: {labelled_any.sum():,} / {len(train):,}')
    print(f'Studies with all available targets non-missing: {labelled_all.sum():,} / {len(train):,}')
    print('Distinct observed values per target (check labels are actually binary):')
    print(y.nunique(dropna=True).to_string())
    print('\nIMPORTANT: do not convert missing labels to 0 without evidence from the competition rules/data documentation.')

print('\n=== REPORT COVERAGE (NO TEXT LABELS GENERATED) ===')
if 'Report' in train.columns:
    report = train['Report'].fillna('').astype(str)
    lengths = report.str.len()
    print('Missing/blank reports:', int((report.str.strip() == '').sum()))
    print('Report character counts:')
    print(lengths.describe(percentiles=[.25, .5, .75, .9, .99]).to_string())
    print('\nExample report excerpts (first 2, truncated; handle medical text responsibly):')
    for i, value in enumerate(report[report.str.strip().ne('')].head(2), 1):
        print(f'[{i}] {value[:350]!r}')

print('\n=== SERIES METADATA DISTRIBUTIONS ===')
for col in ['Fluid_Sensitive', 'Fat_Suppression', 'Anatomical_Plane']:
    if col in series.columns:
        print(f'\n{col}:')
        print(series[col].value_counts(dropna=False).to_string())

print('\n=== TEST / SUBMISSION SCHEMA ===')
for name in ['test.csv', 'test_series.csv', 'sample_submission.csv']:
    if name in found:
        df = pd.read_csv(found[name], nrows=5)
        print(f'\n{name}: columns={df.columns.tolist()} | preview shape={df.shape}')
        if name == 'sample_submission.csv':
            print(df.head(2).to_string(index=False))
            print('Target columns match expected set:', set(TARGETS).issubset(df.columns))

print('\n=== AUDIT COMPLETE ===')
print('Save the printed counts to RESEARCH_LOG.md before modelling. No model was trained and no score was measured.')